# 3D Molecular Equivariance with SE(3)-Transformers

## Table of Contents
* [Introduction](#introduction)
* [The Limits of 2D Molecular Graphs](#2d-limits)
* [Understanding SE(3) Symmetry](#se3-symmetry)
* [Step 1: Generating 3D Conformations with RDKit](#step1)
* [Step 2: Featurizing 3D Molecules with DeepChem](#step2)
* [Step 3: The SE(3)-Transformer Architecture](#step3)
* [Step 4: Empirical Proof of Rotational Invariance](#step4)
* [Conclusion & References](#conclusion)

## Introduction <a class="anchor" id="introduction"></a>

In [Tutorial 1: Introduction to Equivariance](Introduction_to_Equivariance.ipynb), we explored the foundational mathematics of symmetries using toy NumPy arrays. In this tutorial, we take the next step: applying **3D geometric deep learning** to real molecules using DeepChem's native $\mathrm{SE}(3)$-equivariant suite.

You can follow this tutorial using Google Colab by clicking the badge below:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/deepchem/deepchem/blob/master/examples/tutorials/3D_Molecular_Equivariance_with_SE3_Transformers.ipynb)

In [ ]:
# Setup dependencies (uncomment if running in Google Colab)
# !pip install --pre deepchem
# !pip install torch dgl rdkit

import copy
import numpy as np
import torch
import deepchem as dc
from rdkit import Chem
from rdkit.Chem import AllChem

print(f"DeepChem Version: {dc.__version__}")
print(f"PyTorch  Version: {torch.__version__}")

## The Limits of 2D Molecular Graphs <a class="anchor" id="2d-limits"></a>

In traditional graph neural networks (like Graph Convolutional Networks in Tutorial 6), molecules are treated as 2D topological graphs:
- **Nodes** represent atoms (e.g., atomic number, hybridization).
- **Edges** represent chemical bonds (single, double, aromatic).

While 2D graphs work well for topological properties (like basic solubility or molecular weight), they fail when predicting properties that fundamentally depend on **3D spatial conformation**:
1. **Chirality and Stereoisomers:** Two enantiomers share the identical 2D connectivity graph, yet have drastically different biochemical behaviors in receptor binding pockets.
2. **Quantum Chemical Properties:** Dipole moments, vibrational frequencies, and molecular orbital energy gaps (HOMO-LUMO) are continuous functions of precise interatomic distances and dihedral angles.

To model these phenomena accurately, our networks must take 3D coordinates $(x, y, z)$ as inputs.

## Understanding SE(3) Symmetry <a class="anchor" id="se3-symmetry"></a>

When we feed 3D coordinates into a neural network, a major challenge emerges: **coordinate frame ambiguity**.

A molecule in vacuum does not have an absolute orientation. If you rotate a caffeine molecule by $45^\circ$ or translate it $10\text{ \AA}$ to the left, its physical properties (e.g., ground state energy, toxicity) do not change.

The mathematical group governing these transformations is **$\mathrm{SE}(3)$** (Special Euclidean Group in 3 dimensions), which comprises:
- **Translations:** $\mathbb{R}^3$
- **Rotations:** $\mathrm{SO}(3)$ (determinant $+1$ orthogonal matrices)

### Invariance vs. Equivariance
- **Invariance:** A function $f$ is invariant under transformation $g \in G$ if $f(g \cdot x) = f(x)$. Scalar properties like molecular energy must be invariant.
- **Equivariance:** A function $f$ is equivariant under $g \in G$ if $f(g \cdot x) = g \cdot f(x)$. Vector quantities (like atomic forces or dipole moment vectors) must rotate along with the molecule.

DeepChem implements the **$\mathrm{SE}(3)$-Transformer**, an equivariant graph neural network that maintains equivariance through its intermediate layers via spherical harmonics, and collapses them via global pooling into invariant scalar predictions.

## Step 1: Generating 3D Conformations with RDKit <a class="anchor" id="step1"></a>

Let's create an Ethanol molecule (`CCO`) and generate its low-energy 3D conformation using RDKit's ETKDG (Experimental-Torsion Distance Geometry) method.

In [ ]:
# 1. Parse SMILES and add explicit Hydrogens for accurate 3D geometry
smiles = "CCO"
mol = Chem.MolFromSmiles(smiles)
mol = Chem.AddHs(mol)

# 2. Embed 3D coordinates using ETKDG algorithm
embed_status = AllChem.EmbedMolecule(mol, AllChem.ETKDG())
assert embed_status == 0, "3D Embedding failed!"

# Remove Hydrogens if working with heavy-atom representations
mol = Chem.RemoveHs(mol)

print(f"Molecule: {smiles}")
print(f"Number of heavy atoms: {mol.GetNumAtoms()}")
conf = mol.GetConformer()
for i, atom in enumerate(mol.GetAtoms()):
    pos = conf.GetAtomPosition(i)
    print(f"  Atom {i} ({atom.GetSymbol()}): x={pos.x:8.3f}, y={pos.y:8.3f}, z={pos.z:8.3f}")

## Step 2: Featurizing 3D Molecules with DeepChem <a class="anchor" id="step2"></a>

DeepChem provides `EquivariantGraphFeaturizer` to transform 3D molecular structures into geometric graph objects (`GraphData`).

The featurizer extracts:
1. **`node_features`:** One-hot atomic type descriptors (H, C, N, O, F) and atomic numbers.
2. **`node_pos_features`:** The 3D Cartesian coordinates $\mathbf{x}_i \in \mathbb{R}^3$.
3. **`edge_features`:** Relative displacement vectors $(\mathbf{x}_i - \mathbf{x}_j)$ between bonded atoms.
4. **`edge_weights`:** Binned pairwise Euclidean distances.

In [ ]:
# Initialize DeepChem's EquivariantGraphFeaturizer
featurizer = dc.feat.EquivariantGraphFeaturizer(fully_connected=False, embeded=True)

# Featurize the molecule
graph = featurizer.featurize([mol])[0]

print(f"Node features shape:      {graph.node_features.shape}")
print(f"3D Node positions shape:  {graph.node_pos_features.shape}")
print(f"Edge index shape:         {graph.edge_index.shape}")
print(f"Edge displacement shape:  {graph.edge_features.shape}")
print(f"Edge distance bins shape: {graph.edge_weights.shape}")

### Computing the Equivariant Spherical Harmonics Basis

Under the hood, the $\mathrm{SE}(3)$-Transformer represents directional edge features using spherical harmonics $Y_l^m(\hat{\mathbf{r}}_{ij})$. 

Let's convert our `GraphData` into a DGL graph and inspect the computed basis up to degree $L=3$.

In [ ]:
from deepchem.utils.equivariance_utils import get_equivariant_basis_and_r

# Convert GraphData to DGL Graph
g = graph.to_dgl_graph()
g.edata['w'] = torch.tensor(graph.edge_weights, dtype=torch.float32)

# Compute the irreducible spherical harmonic basis up to degree L=3
basis, r = get_equivariant_basis_and_r(g, max_degree=3)

print(f"Total directional graph edges: {g.num_edges()}")
print(f"Total basis representations computed: {len(basis)} (sum over (2l + 1) for l in 0..3: (3+1)^2 = 16)")

## Step 3: The SE(3)-Transformer Architecture <a class="anchor" id="step3"></a>

The `SE3TransformerModel` in DeepChem wraps the $\mathrm{SE}(3)$-Transformer in a standard `TorchModel` interface.

Key hyper-parameters:
- `num_layers`: Number of equivariant self-attention layers.
- `atom_feature_size`: Dimension of scalar input features (here 6, from our featurizer).
- `num_degrees`: Maximum degree $L$ of spherical tensors to propagate.
- `n_heads`: Number of equivariant attention heads.
- `pooling`: Global node aggregation strategy (`'max'` or `'avg'`) to produce an invariant scalar property.

In [ ]:
from deepchem.models.torch_models.se3_transformer import SE3TransformerModel

# Instantiate SE(3)-Transformer Model
model = SE3TransformerModel(
    num_layers=1,
    atom_feature_size=6,
    num_channels=8,
    num_degrees=2,
    edge_dim=4,
    pooling='max',
    n_heads=1,
    num_workers=0
)

print("Model successfully initialized:")
print(model.model)

## Step 4: Empirical Proof of Rotational Invariance <a class="anchor" id="step4"></a>

Now we perform the critical scientific verification:
1. We feed the original 3D molecule conformation into `SE3TransformerModel`.
2. We construct a 3D rotation matrix $R \in \mathrm{SO}(3)$ (a $90^\circ$ rotation around the Z-axis):
   $$R = \begin{bmatrix} \cos\theta & -\sin\theta & 0 \\ \sin\theta & \cos\theta & 0 \\ 0 & 0 & 1 \end{bmatrix}$$
3. We rotate the 3D atomic coordinates $\mathbf{X}_{\text{rot}} = \mathbf{X} \cdot R^T$ and displacement vectors.
4. We evaluate the model on the rotated conformation and compare the predicted property.

In [ ]:
# 1. Construct original dataset
features_orig = graph
dataset_orig = dc.data.NumpyDataset(X=np.array([features_orig]), y=np.array([[0.0]]))

# 2. Build 3D rotation matrix (90 degrees around Z-axis)
theta = np.pi / 2
R = np.array([
    [np.cos(theta), -np.sin(theta), 0.0],
    [np.sin(theta),  np.cos(theta), 0.0],
    [0.0,            0.0,           1.0]
], dtype=np.float32)

# 3. Create rotated features
features_rot = copy.deepcopy(features_orig)
features_rot.node_pos_features = np.dot(features_orig.node_pos_features, R)
features_rot.edge_features = np.dot(features_orig.edge_features, R)
dataset_rot = dc.data.NumpyDataset(X=np.array([features_rot]), y=np.array([[0.0]]))

# 4. Run model inference on both conformations
pred_orig = model.predict(dataset_orig)[0, 0]
pred_rot  = model.predict(dataset_rot)[0, 0]

diff = abs(pred_orig - pred_rot)

print("=" * 60)
print(f"Predicted Property (Original 3D Pose): {pred_orig:.8f}")
print(f"Predicted Property (Rotated  3D Pose): {pred_rot:.8f}")
print(f"Absolute Discrepancy:                 {diff:.10f}")
print("=" * 60)

assert np.isclose(pred_orig, pred_rot, atol=1e-5), "Invariance check failed!"
print("SUCCESS: Model prediction is mathematically invariant to 3D spatial rotation!")

### Why This Matters
A standard dense feed-forward network or multi-layer perceptron taking flattened 3D coordinates $(x_1, y_1, z_1, \dots, x_N, y_N, z_N)$ produces completely different numbers when the molecule is rotated. Standard models require augmenting the dataset with thousands of random rotational duplicates to learn approximate invariance.

In contrast, the **$\mathrm{SE}(3)$-Transformer guarantees rotational invariance exact to floating-point precision ($< 10^{-7}$)** by mathematical construction. This dramatic inductive bias allows equivariant models to achieve high data efficiency on small chemical datasets like QM9.

## Conclusion & References <a class="anchor" id="conclusion"></a>

In this tutorial, you learned:
1. Why 3D spatial representations are essential for modeling geometric molecular properties.
2. How to use `EquivariantGraphFeaturizer` to extract 3D Cartesian coordinates, pairwise displacement vectors, and distance bins.
3. How to initialize and run DeepChem's native `SE3TransformerModel`.
4. How $\mathrm{SE}(3)$ equivariance ensures consistent scalar property predictions under arbitrary spatial rotations.

### References
1. Fuchs, F. B., Worrall, D. E., Fischer, V., & Welling, M. (2020). *SE(3)-Transformers: 3D Roto-Translation Equivariant Attention Networks*. Advances in Neural Information Processing Systems (NeurIPS 2020).
2. Siguenza, J., & Ramsundar, B. (2025). *DeepChem Equivariant: SE(3)-Equivariant Support in an Open-Source Molecular Machine Learning Library*.

# Congratulations! Time to join the Community!

Congratulations on completing this tutorial notebook! If you enjoyed working through the tutorial and want to continue working with DeepChem, we encourage you to explore the rest of the tutorials in this series. You can also help the DeepChem community in the following ways:

## Star DeepChem on [GitHub](https://github.com/deepchem/deepchem)
This helps build awareness of the DeepChem project and the tools for open-source drug discovery that we are building.

## Join the DeepChem Community
- Join the [DeepChem Discord](https://discord.gg/RYTrUY8Ssn) to chat with researchers and developers.
- Participate in discussions on the [DeepChem Forum](https://forum.deepchem.io/).